<a href="https://colab.research.google.com/github/Praise-Oluwasikemi/TS-Academy-Group-4-IBM-HR-Analytics-Attrition/blob/main/group_04_yourname.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Submission Summary

| Field | Value |
|---|---|
| Group number | 04 |
| Your full name | PRAISE ADETAYO |
| Dataset | IBM HR Analytics Attrition |
| Depth track | Classification Depth |
| Team members | DAVID WEALTH, MICHAEL SAMPSON, OKOGBO JOSEPH, VITOR ODURONBI, HELEN EMIEWO, POPOOLA TEMIDAYO, OGUNRIONOLA CHRISTANAH, ADENMOSUN BETHEL, DORCAS YUSUF, ALLI OLAMILEKAN, PRAISE ADETAYO |
| Final model | Logistic Regression, decision threshold 0.30 |
| Stratified dummy F1 / ROC AUC | 0.141 / 0.486 |
| Final model F1 / ROC AUC | 0.567 / 0.832 |
| Leakage columns dropped | None found |
| Full run time on Colab | *** filled in at the end *** |

# What this project is about

A company has records for 1,470 of its employees. For each person we know things
like their age, salary, job, how far they live from the office, how long they have
been there, and how they rated their satisfaction at work. We also know one more
thing about each of them: whether they resigned or stayed (ATTRITION).

The company wants two things from this data. First, an honest picture of what
natural types of employee exist here. Second, something that can spot the people
likely to resign, before they do.

That is what this notebook does, working through it in seven parts. Our explanation as a group is written in a way that
someone who has never seen code before can read it from top to bottom and
follow what happened and why.

## Part 1. Data Foundation

Before building anything, understand the data.

### 1.0 Getting the tools ready

This first block loads all the software tools the notebook will use. Nothing
interesting happens here, but it has to run first or nothing else works.

In [ ]:
# Install the one extra package Colab does not come with.
# kagglehub is what lets this notebook download the dataset by itself.
!pip install -q kagglehub imbalanced-learn

# This start a stopwatch so as to report at the end how long the whole notebook took to run.
import time
RUN_START = time.time()

# Tools for handling files and databases
import os, sqlite3, warnings

# Tools for working with tables of data and doing maths on them
import numpy as np
import pandas as pd

# Tools for drawing charts
import matplotlib.pyplot as plt
import seaborn as sns

# The tool that downloads the dataset from Kaggle
import kagglehub

# Machine learning tools
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# Hide warning messages that clutter the output but do not affect results
warnings.filterwarnings('ignore')

# Fix the random seed so that all group members re-running this notebook gets the same number
#Without this, results shift slightly every run.
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Chart styling and an instruction to show all columns when printing tables
sns.set_style('whitegrid')
pd.set_option('display.max_columns', 100)

print('All libraries loaded. Ready to start.')

### 1.1 Getting the data and putting it into a database

The dataset is downloaded by the notebook itself rather than uploaded by hand, so it
reproduces on any fresh machine. It is then copied into a small SQLite database so
the questions below can be asked in SQL.

In [ ]:
# Download the dataset straight from Kaggle.
path = kagglehub.dataset_download('pavansubhasht/ibm-hr-analytics-attrition-dataset')

# Find the CSV file inside the downloaded folder and read it into a table
csv_file = [f for f in os.listdir(path) if f.endswith('.csv')][0]
df_raw = pd.read_csv(os.path.join(path, csv_file))

# Copy that table into a small database file to ask questions using SQL
conn = sqlite3.connect('hr.db')
df_raw.to_sql('employees', conn, if_exists='replace', index=False)

print('File downloaded:', csv_file)
print('The data has', df_raw.shape[0], 'employees and', df_raw.shape[1], 'columns.')
print('Rows now stored in the database:',
      pd.read_sql('SELECT COUNT(*) AS n FROM employees', conn).iloc[0, 0])

# show the first five employees so we can see what the data looks like
df_raw.head()

### 1.2 Three questions for the data

Three questions an HR manager with a limited retention budget would need answered.

#### Question 1: Where are we losing people?

We asked this in two steps. First by department, because there are only three of
them, so each one has plenty of people in it and the percentages can be trusted.
Then we looked inside the biggest department at the individual job roles, because
an average across a whole department can hide a problem sitting in one particular
role.

In [ ]:
# Count how many people work in each department, how many of them left,
# and turn that into a percentage.
q1a = """
SELECT Department,
       COUNT(*) AS total_staff,
       SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS quit_count,
       ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS quit_pct
FROM employees
GROUP BY Department
ORDER BY quit_pct DESC
"""
pd.read_sql(q1a, conn)

Now the second step, inside Research & Development.

In [ ]:
# The same question again, but looking only inside Research & Development,
# broken down by the individual job roles within it.
q1b = """
SELECT JobRole,
       COUNT(*) AS total_staff,
       SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS quit_count,
       ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS quit_pct
FROM employees
WHERE Department = 'Research & Development'
GROUP BY JobRole
ORDER BY quit_pct DESC
"""
pd.read_sql(q1b, conn)

**What was found**

Sales exhibits the highest departmental turnover, losing 20.6% of its staff. Research and Development appears healthier at 13.8%, while Human Resources sits at 19% across a small sample size of 63 employees (where minor headcount changes significantly shift the overall percentage).

Further granular breakdown reveals a critical insight: within Research and Development, Laboratory Technicians suffer a high attrition rate of 23.9% across 259 employees—exceeding even the Sales department. This risk was completely hidden by the overall departmental average of 13.8%.

Conclusion: Attrition challenges are concentrated in Sales overall, alongside a specific, high-risk operational role in Laboratory Technicians within R&D.

What this data cannot tell us is why. There is no column recording anyone's
reasons. It might be workload, staffing levels, the number of samples each
technician handles, or something nobody has thought of. That would have to be found
out by asking them directly.

#### Question 2: Do people who work overtime leave more?

In [ ]:
# Compare people who work overtime against people who do not,
# and see what share of each group left.
q2 = """
SELECT OverTime,
       COUNT(*) AS total_staff,
       SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) AS quit_count,
       ROUND(100.0 * SUM(CASE WHEN Attrition = 'Yes' THEN 1 ELSE 0 END) / COUNT(*), 1) AS quit_pct
FROM employees
GROUP BY OverTime
ORDER BY quit_pct DESC
"""
pd.read_sql(q2, conn)

**What we found**

Overtime staff leave at 30.5% against 10.4% for everyone else, a three-fold
difference and the largest gap in the dataset.

Correlation is strong but causality cannot be proven without operational context such
as team-level workload. Targeted hiring to redistribute work is still the most
actionable response, treated as a change worth measuring rather than a certain fix.

#### Question 3: Do the people who leave earn less than the people who stay?

In [ ]:
# For each department, compare the average salary of the people who stayed
# against the average salary of the people who left.
q3 = """
SELECT Department,
       Attrition,
       COUNT(*) AS staff,
       ROUND(AVG(MonthlyIncome), 0) AS avg_monthly_income
FROM employees
GROUP BY Department, Attrition
ORDER BY Department, Attrition
"""
pd.read_sql(q3, conn)

**What was found**

In Human Resources and R&D, leavers earned roughly half what stayers earned. The gap
is much smaller in Sales.

A full salary review is likely unaffordable. Targeted bonuses for the lowest-paid
bands cost far less and reach the group most at risk.

One complication remains open: the lowest earners are also the youngest and newest
staff, who move jobs more often regardless of pay. Part 2 builds a feature
specifically to separate the two effects.

### 1.3 Looking at the data as a whole

In [ ]:
# The basic facts about the dataset before changing anything.

print('Number of employees and number of columns:', df_raw.shape)
print()

print('What kind of information each column holds:')
print(df_raw.dtypes.value_counts())
print()

# Check whether any information is missing anywhere in the table
print('Total number of empty cells in the whole dataset:', df_raw.isnull().sum().sum())
print()

# Count how many people left and how many stayed
print('How many people left versus stayed:')
print(df_raw['Attrition'].value_counts())
print()
print('The same thing as a proportion:')
print(df_raw['Attrition'].value_counts(normalize=True).round(4))

**What the basic numbers tell us**

1,470 employees, 35 columns, zero missing values. 237 left and 1,233 stayed, so the
positive class is 16.12% of the data.

Because the classes are imbalanced, raw accuracy is misleading: predicting "nobody
resigns" for everyone scores 83.88% while identifying not a single resignation risk.
Evaluation must therefore focus on recall, F1 and ROC AUC for the positive class,
which is what Part 4 reports and what Part 5 exists to improve.

At 1,470 rows the dataset is also below the 2,000 threshold, so Part 4 uses stratified
cross-validation rather than a single train/test split.

In [ ]:
# Five charts. Red means the employee left, blue means they stayed.

fig, ax = plt.subplots(2, 3, figsize=(18, 9))
colours = {'No': '#4C72B0', 'Yes': '#C44E52'}

# The overall quit rate, used as a reference line on the bottom charts
overall = 100 * (df_raw['Attrition'] == 'Yes').mean()

# Top row: three columns that are ordinary numbers, showing how leavers and
# stayers are spread across each one.
for i, col in enumerate(['Age', 'MonthlyIncome', 'YearsAtCompany']):
    sns.histplot(data=df_raw, x=col, hue='Attrition', bins=25,
                 palette=colours, alpha=0.6, ax=ax[0, i])
    ax[0, i].set_title(col)

# Bottom left two: quit rate for each category, with a dashed line showing the
# company average so it is obvious which groups are worse than normal.
for i, col in enumerate(['OverTime', 'BusinessTravel']):
    rates = df_raw.groupby(col)['Attrition'].apply(lambda s: 100 * (s == 'Yes').mean()).sort_values()
    rates.plot(kind='barh', color='#C44E52', ax=ax[1, i])
    ax[1, i].axvline(overall, ls='--', c='k', lw=1)
    ax[1, i].set_title(f'{col} - quit rate by group (dashed line = company average {overall:.1f}%)')
    ax[1, i].set_xlabel('quit %')

# Bottom right: how the quit rate changes with each extra year at the company.
tenure = df_raw.groupby('YearsAtCompany')['Attrition'].apply(lambda s: 100 * (s == 'Yes').mean())
tenure[tenure.index <= 10].plot(marker='o', color='#C44E52', ax=ax[1, 2])
ax[1, 2].axhline(overall, ls='--', c='k', lw=1)
ax[1, 2].set_title('Quit % by number of years at the company')
ax[1, 2].set_ylabel('quit %')

plt.tight_layout()
plt.show()

**What the charts show**

Red is leavers, blue is stayers. In the top three charts red concentrates on the left:
leavers are younger, paid less, and newer. Red is also far smaller everywhere, which
is the 16% imbalance made visible.

Against the 16.1% company average (dashed line), overtime and frequent travel both sit
well above. Frequent travellers leave at 24.9% against 8.0% for non-travellers.

Tenure shows the sharpest pattern: 34.9% leave in year one, 21.3% in year two, 8.1%
after ten years. Age, salary and tenure are not three findings but one group seen three
ways, which is exactly why Question 3 could not be answered alone.


### 1.4 Cleaning the data

Four columns are removed because they carry no usable information, missing values are
checked rather than assumed, the data is screened for leakage, and the target is
converted to numbers. No blanket `fillna` is applied and no rows are dropped. Each
decision is justified underneath.

In [ ]:
# Work on a copy so the original downloaded data stays untouched.
df = df_raw.copy()

# --- Step 1: check for columns that hold no useful information ----------------
# A column where everybody has the same value cannot explain why some people
# left and others did not.
print('Checking four suspicious columns:')
for col in ['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']:
    print('  %-16s number of different values: %5d' % (col, df[col].nunique()))

df = df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber'])

# --- Step 2: check for missing information ------------------------------------
print()
print('Number of columns with any missing values:', (df.isnull().sum() > 0).sum())
print('Nothing is missing, so nothing needs to be filled in or removed.')

# --- Step 3: check for columns that give away the answer -----------------------
# We looked through all 35 original columns for anything that would only be known
# after somebody had already resigned, such as a leaving date or an exit
# interview score. There is nothing like that in this dataset.

# --- Step 4: turn the answer column into numbers -------------------------------
# Models cannot read the words "Yes" and "No", so from this point on
# 1 means the person left and 0 means they stayed.
df['Attrition'] = df['Attrition'].map({'Yes': 1, 'No': 0})

print()
print('Columns remaining after cleaning:', df.shape[1])
print('Share of employees who left:', round(df['Attrition'].mean(), 4))

**Why these changes**

**Four columns removed.** EmployeeCount, StandardHours and Over18 hold identical
zero-variance values across all 1,470 records, so they cannot explain an outcome that
varies. EmployeeNumber is an arbitrary staff ID with 1,470 unique values, which risks
memorisation rather than learning.

**Zero missing values**, verified rather than assumed, so no imputation or row deletion
was performed.

**No leakage found.** A leaking column is one only knowable after somebody resigned, such
as a termination date or exit-interview score. All 35 original columns describe the
employee while still employed. Worth stating explicitly, as other datasets in this
capstone do contain such columns.

**Attrition converted to numbers**, 1 meaning the person left. Everything downstream
treats it as 1/0, which matters in Part 2.


## Part 2. Feature Engineering and Encoding (20%)


### 2.1 Three new columns

We construct three domain-specific features to capture employee turnover dynamics:

1. **Income_Per_Job_Level** : Captures internal pay equity by evaluating an employee's
   compensation relative to their job seniority.
2. **Burnout_Risk** : An interaction feature combining daily commute
   (DistanceFromHome) with OverTime work, identifying employees at high risk of burnout.
3. **Company_Tenure_Ratio** : Measures career loyalty versus job-hopping history by
   evaluating company tenure against total working years.

In [ ]:
# Copy of the cleaned table from Part 1
df_fe = df.copy()

# Feature 1: pay relative to seniority, so a junior on a junior salary
# is not mistaken for someone who is underpaid
df_fe['Income_Per_Job_Level'] = df_fe['MonthlyIncome'] / df_fe['JobLevel']

# Feature 2: overtime and commute distance combined into one strain score
overtime_numeric = (df_fe['OverTime'] == 'Yes').astype(int)
df_fe['Burnout_Risk'] = df_fe['DistanceFromHome'] * overtime_numeric

# Feature 3: share of a whole working life spent at this company
# (+1 in the denominator avoids dividing by zero)
df_fe['Company_Tenure_Ratio'] = df_fe['YearsAtCompany'] / (df_fe['TotalWorkingYears'] + 1)

print('Shape after adding three features:', df_fe.shape)
df_fe[['Income_Per_Job_Level', 'Burnout_Risk', 'Company_Tenure_Ratio']].head()

**Checking the three features separate leavers from stayers**

A feature is only worth keeping if its quit rate varies across groups.

In [ ]:
# Check each feature separates leavers from stayers before keeping it.
# Split into four equal-sized groups and compare quit rates.

for col in ['Income_Per_Job_Level', 'Company_Tenure_Ratio']:
    groups = pd.qcut(df_fe[col], 4, duplicates='drop')
    table = df_fe.groupby(groups, observed=True)['Attrition'].agg(
        staff='size', quit_pct=lambda s: round(100 * s.mean(), 1))
    print(col, '(lowest group first)')
    print(table.to_string())
    print()

# Burnout_Risk handled separately: most of the company scores exactly 0.
bands = pd.cut(df_fe['Burnout_Risk'], [-1, 0, 5, 12, 30],
               labels=['0 (no overtime)', '1-5', '6-12', '13+'])
print('Burnout_Risk')
print(df_fe.groupby(bands, observed=True)['Attrition'].agg(
    staff='size', quit_pct=lambda s: round(100 * s.mean(), 1)).to_string())
print()

print('OverTime on its own, for comparison:')
print(df_fe.groupby('OverTime')['Attrition'].agg(
    staff='size', quit_pct=lambda s: round(100 * s.mean(), 1)).to_string())

**Results**

**Income_Per_Job_Level** is the strongest: 21.5% → 17.7% → 15.5% → 9.8%. The
lowest-paid-for-their-level leave at more than twice the rate of the highest.

**Burnout_Risk** adds information beyond overtime alone. Overtime by itself splits the
company 10.4% / 30.5%; this feature splits the overtime group further into 26.7%,
24.6% and 41.8%, so commute distance is genuinely contributing. Note the limitation:
because the score is distance × overtime, all 1,054 non-overtime staff score 0
regardless of commute, so it measures overtime intensified by distance rather than
strain in general.

**Company_Tenure_Ratio** is the weakest: 17.3%, 23.2%, 12.5%, 11.4%. The two
longest-committed groups clearly leave less, but the second group is the highest of the
four rather than the first. We have no confident explanation for that and record it as
an open observation rather than inventing one.

### 2.2 Separating the target from the features

Attrition was already converted to 1/0 in Part 1, so it is taken as it is. The printed
check exists so that any upstream change breaking the target is caught here rather than
in Part 4.

In [ ]:
# X = what we know about each employee.  y = did they leave.
X = df_fe.drop(columns=['Attrition'])

# Attrition was already converted to 1/0 in Part 1, so it is used as it is.
y = df_fe['Attrition']

# Guard: if this ever prints 0 leavers, the target has been broken upstream
# and nothing in Parts 4 and 5 can work.
print('Employees:', len(y))
print('Leavers in y:', int(y.sum()), '  (must be 237)')
print('Values in y:', sorted(y.unique()), '  (must be [0, 1])')

### 2.3 Encoding

Categorical variables are converted into numerical representations:

* **Ordinal Encoding**: Applied to BusinessTravel because it possesses an inherent
  logical order (Non-Travel < Travel_Rarely < Travel_Frequently), confirmed by the Part 1
  quit-rate ladder of 8.0%, 15.0% and 24.9%.
* **One-Hot Encoding (pd.get_dummies)**: Applied to nominal variables without order
  (Department, EducationField, JobRole, MaritalStatus, Gender, OverTime).
* **drop_first=True**: Applied to one-hot encoding to eliminate linear redundancy
  (multicollinearity / dummy variable trap).
* **No encoding**: Education, JobLevel, StockOptionLevel, TrainingTimesLastYear, the
  five satisfaction scores and PerformanceRating are already ordered numbers on a fixed
  scale, verified in the output below.

In [ ]:
# BusinessTravel has a real order, so it is numbered rather than split.
# Part 1 supports this: quit rates climb cleanly 8.0% -> 15.0% -> 24.9%.

travel_map = {'Non-Travel': 0, 'Travel_Rarely': 1, 'Travel_Frequently': 2}
if 'BusinessTravel' in X.columns:
    X['BusinessTravel'] = X['BusinessTravel'].map(travel_map)

print('BusinessTravel is now numbered:', sorted(X['BusinessTravel'].unique()))

Identifying the text columns that still need encoding:

In [ ]:
# Remaining text columns. No natural order, so each is split into yes/no columns.

cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
print('Text columns still to encode:', cat_cols)
X[cat_cols].head()

One-hot encoding with `drop_first=True`, which removes one column per set because
it can always be inferred from the others and would otherwise introduce the dummy
variable trap.


In [ ]:
# drop_first=True removes one column per set: it can always be worked out
# from the others, so keeping it is pure redundancy.

X_encoded = pd.get_dummies(X, columns=cat_cols, drop_first=True)

print('Shape after encoding:', X_encoded.shape)
print('Text columns remaining:', X_encoded.select_dtypes(include='object').shape[1])
print()
print('Already ordered numbers, so no encoding needed:')
for c in ['Education', 'JobLevel', 'StockOptionLevel', 'TrainingTimesLastYear',
          'EnvironmentSatisfaction', 'JobSatisfaction', 'RelationshipSatisfaction',
          'JobInvolvement', 'WorkLifeBalance', 'PerformanceRating']:
    print('  %-26s values: %s' % (c, [int(v) for v in sorted(X_encoded[c].unique())]))

### 2.4 Scaling

We applied **StandardScaler** (z-score standardization) across the feature columns
because the models we go on to use — K-Means clustering in Part 3 and regularized
classifiers in Part 4 — rely on distance calculations and are therefore sensitive to
variable scale. MonthlyIncome runs to 20,000 while JobSatisfaction runs 1 to 4, so
without scaling the former would dominate purely through its units.

StandardScaler transforms features to mean 0 and variance 1 without bounding extreme
values into arbitrary limits. MinMaxScaler was rejected for that reason: the small
number of executives near 20,000 would compress the majority of staff into a narrow
band at the bottom of the range, where real differences become hard to distinguish.

In [ ]:
# Put every column on the same footing so none dominates just because
# its numbers are larger.

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_encoded)

# Keep a named version: Parts 3 and 4 need column names for cluster
# profiles and feature importances.
X_scaled_df = pd.DataFrame(X_scaled, columns=X_encoded.columns)

print('Employees:', X_scaled_df.shape[0], ' Columns describing them:', X_scaled_df.shape[1])
print()
print('What scaling did to two columns with very different ranges:')
print(pd.DataFrame({
    'average before': X_encoded[['MonthlyIncome', 'JobSatisfaction']].mean().round(2),
    'spread before': X_encoded[['MonthlyIncome', 'JobSatisfaction']].std().round(2),
    'average after': X_scaled_df[['MonthlyIncome', 'JobSatisfaction']].mean().round(2),
    'spread after': X_scaled_df[['MonthlyIncome', 'JobSatisfaction']].std().round(2)}))


Part 2 complete: the dataset is clean, carries three tested features, is fully numeric
and scaled, and is ready for clustering and modelling.

## Part 3. Clustering Analysis (20%)

The company wants an honest picture of what natural types of employee exist here —
not categories we invent, but groups the data itself suggests. This part uses a
method called K-Means clustering to let the computer find those groups on its own,
based on things like age, pay, tenure, and satisfaction. We then check whether some
groups leave the company more than others

### 3.1 Using the official feature set from Part 2

Clustering works by measuring how similar or different employees are, so it needs
pure numbers to compare. Rather than picking our own smaller set, we use the full
feature set handed off from Part 2: `X_scaled_df`, 46 columns covering demographics,
pay, tenure, satisfaction, and the three engineered features
(Income_Per_Job_Level, Burnout_Risk, Company_Tenure_Ratio), already cleaned, encoded
and scaled.

In [ ]:
# Part 2 built X_scaled_df: 46 columns, fully encoded and scaled, including
# three new engineered features (Income_Per_Job_Level, Burnout_Risk,
# Company_Tenure_Ratio). We use this directly instead of picking our own
# smaller set, so clustering reflects the team's full, agreed feature set.

X_cluster = X_scaled_df.values   # same data, as a plain array for KMeans and PCA

print('Features used for clustering:', X_scaled_df.shape[1])
print('Employees:', X_scaled_df.shape[0])

### 3.2 Elbow Method

We don't get to choose an arbitrary number of groups — we need to find the number
that fits the data. The elbow method tries splitting employees into 2, 3, 4 and so on
up to 10 groups, and for each attempt measures how tightly packed each group is. More
groups always makes this number look better, but at some point the improvement barely
matters anymore. That turning point, the elbow, hints at a sensible number of groups.

In [ ]:
# We try K = 2 through 10 groups. For each one, we measure "inertia" -
# how tight each group is (how close employees sit to their group's center).
# More groups always improves this score, but at some point the improvement
# stops being worth it. That point is the "elbow" - where the line bends.

inertia_values = []
k_range = range(2, 11)

for k in k_range:
    kmeans_test = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    kmeans_test.fit(X_cluster)
    inertia_values.append(kmeans_test.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia_values, marker='o', color='#4C72B0')
plt.title('Elbow Method - look for where the line bends')
plt.xlabel('Number of groups (K)')
plt.ylabel('Inertia (tightness of groups)')
plt.xticks(list(k_range))
plt.show()

# Print the actual numbers so we can read them precisely, not just eyeball the chart
for k, inertia in zip(k_range, inertia_values):
    print(f'K={k}: inertia={inertia:.1f}')

**What was found**

The line falls smoothly with no sharp bend anywhere. Inertia drops 7.1% from K=2 to
K=3, then roughly 3% for each group added after that. There is no clear elbow to
read, so this test on its own does not settle the question and we need the second
signal below.

### 3.3 Silhouette Score

The elbow chart was ambiguous, so we use a second, more rigorous test. For every
employee, the silhouette score checks how close they are to others in their own group
compared to how close they are to the nearest other group. It produces one number per
K, from -1 to 1, where higher means the groups are more clearly separated.

In [ ]:
# The elbow chart can be a bit subjective to read by eye, so this gives a
# more rigorous, single-number check. For each K, it measures how well
# each employee fits their own group versus how close they are to the
# nearest other group. Score ranges from -1 to 1 - higher means the
# groups are clearly separated from each other.

silhouette_values = []

for k in k_range:
    kmeans_test = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels = kmeans_test.fit_predict(X_cluster)
    silhouette_values.append(silhouette_score(X_cluster, labels))

plt.figure(figsize=(8, 5))
plt.plot(k_range, silhouette_values, marker='o', color='#C44E52')
plt.title('Silhouette Score - higher is better')
plt.xlabel('Number of groups (K)')
plt.ylabel('Silhouette Score')
plt.xticks(list(k_range))
plt.show()

for k, sil in zip(k_range, silhouette_values):
    print(f'K={k}: silhouette={sil:.3f}')

**What was found**

K=2 scores 0.126, the highest of any option tested, even with the full 46-feature
set. Every other value stays in the 0.086–0.107 range, with a minor secondary bump at
K=8 that still falls short. On this signal the answer is unambiguous: two groups
produce the best-separated split, and we proceed with K=2.

One caveat we want to state plainly is that every
silhouette score here is low in absolute terms — 0.126 is much closer to 0 than to
the 0.5 that would indicate clearly distinct groups. Combined with the absent elbow,
this tells us the workforce does not fall into sharply separated natural types. It is
closer to one continuous population, and K=2 is the best available dividing line
through it rather than a boundary that was already there. The clusters are still
useful, as the profiles below show, but we would not want to overstate how distinct
they are.

### 3.4 Running K-Means with the chosen K

With K=2 confirmed as the best-supported choice, we run K-Means one final time and
give every employee a label, Cluster 0 or Cluster 1, showing which group they were
placed in.

In [ ]:
# The silhouette score told us K=2 gives the cleanest, most well-separated
# groups. Now we actually run K-Means with that setting and give every
# employee a label: which of the 2 groups they belong to.

FINAL_K = 2

kmeans_final = KMeans(n_clusters=FINAL_K, random_state=RANDOM_STATE, n_init=10)
df['Cluster'] = kmeans_final.fit_predict(X_cluster)
df_fe['Cluster'] = df['Cluster']      # same labels on the readable table

print('Employees per cluster:')
print(df['Cluster'].value_counts())
print()
print('Share of employees in each cluster:')
print(df['Cluster'].value_counts(normalize=True).round(3))

**What was found**

The two groups are unevenly sized: Cluster 0 holds 326 employees (22.2%) and Cluster 1
holds 1,144 (77.8%). This lopsided minority/majority pattern holds up even after
adding the three engineered features and full encoding.

### 3.5 PCA Visualization

Our clustering used 46 columns at once, which cannot be drawn on a normal chart. PCA
(Principal Component Analysis) compresses those columns down into 2 summary
dimensions that capture as much of the original spread as possible, so we can plot
every employee as a dot and see whether the two clusters actually look separated
rather than just trusting the numbers.

In [ ]:
# Our clustering used 46 columns at once, which we cannot draw on a normal
# chart. PCA compresses those 46 columns down into just 2 "summary" columns
# that capture as much of the original spread in the data as possible, so
# we can plot every employee as a single dot and see the two groups visually.

pca = PCA(n_components=2, random_state=RANDOM_STATE)
pca_coords = pca.fit_transform(X_cluster)

df['PCA1'] = pca_coords[:, 0]
df['PCA2'] = pca_coords[:, 1]

print('Variance captured by each PCA dimension:', pca.explained_variance_ratio_.round(3))
print('Total variance captured:', round(pca.explained_variance_ratio_.sum(), 3))

plt.figure(figsize=(8, 6))
colors_cluster = {0: '#C44E52', 1: '#4C72B0'}
for cluster_id in sorted(df['Cluster'].unique()):
    subset = df[df['Cluster'] == cluster_id]
    plt.scatter(subset['PCA1'], subset['PCA2'],
                label=f'Cluster {cluster_id}', alpha=0.6,
                color=colors_cluster[cluster_id])

plt.title('Employee Clusters (compressed to 2 dimensions using PCA)')
plt.xlabel('PCA Dimension 1')
plt.ylabel('PCA Dimension 2')
plt.legend()
plt.show()

# How much do the two groups actually overlap along the main PCA axis?
pc1_range_c1 = (df.loc[df.Cluster == 1, 'PCA1'].min(), df.loc[df.Cluster == 1, 'PCA1'].max())
inside = df.loc[df.Cluster == 0, 'PCA1'].between(*pc1_range_c1).mean()
print(f'Share of Cluster 0 employees falling inside Cluster 1 range on PCA1: {inside:.1%}')

**What was found**

The two PCA dimensions capture only 19.9% of the original information (12.3% and
7.5%). Spreading 46 features across just 2 dimensions keeps only a fraction of any
single one, so this chart is a flattened shadow of the real structure rather than a
faithful picture of it.

On the chart the two clusters do separate visibly along the horizontal axis, with
Cluster 0 sitting to the right. But the measured overlap tells a more careful story:
**26.7% of Cluster 0 employees fall inside Cluster 1's range** on that axis, so the
boundary is a region rather than a gap. That is consistent with the low silhouette
score of 0.126 rather than contradicting it.

We are deliberately not claiming the chart proves a strong distinction. It shows a
real tendency in the two directions we can draw, and two groups that look separated
here could still overlap heavily in the 44 directions this chart cannot show. The
honest reading is that the split is genuine but soft.

### 3.6 Cluster Profiles

Knowing two groups exist is not useful on its own — we need to know what actually
distinguishes them, and whether one group leaves the company more than the other.
This step averages the readable features within each cluster and computes each
cluster's attrition rate.

In [ ]:
# We profile using the original, human-readable columns from df_fe
# (before encoding), plus Attrition, so the table stays interpretable -
# reading averages across 46 one-hot encoded columns would be unreadable.

profile_columns = [
    'Age', 'MonthlyIncome', 'JobLevel', 'DistanceFromHome', 'YearsAtCompany',
    'YearsInCurrentRole', 'TotalWorkingYears', 'NumCompaniesWorked',
    'JobSatisfaction', 'EnvironmentSatisfaction', 'WorkLifeBalance',
    'JobInvolvement', 'RelationshipSatisfaction',
    'Income_Per_Job_Level', 'Burnout_Risk', 'Company_Tenure_Ratio'
]

cluster_profile = df_fe.groupby('Cluster')[profile_columns].mean().round(2)
cluster_profile['Employees'] = df_fe.groupby('Cluster').size()

# Attrition is computed from the raw column, NOT from the rounded table above,
# so the percentage is exact.
cluster_profile['AttritionRate(%)'] = (df_fe.groupby('Cluster')['Attrition'].mean() * 100).round(1)

print('Most common job roles in each cluster:')
for c in sorted(df_fe['Cluster'].unique()):
    top = df_fe.loc[df_fe['Cluster'] == c, 'JobRole'].value_counts().head(3)
    print(f'  Cluster {c}: ' + ', '.join(f'{r} ({n})' for r, n in top.items()))
print()
print('Average profile of each cluster:')
cluster_profile.T

**What was found**

**Cluster 0 (326 employees)** is established, senior staff: average age 44.8, average
monthly income 13,541, 13.7 years at the company, average job level 3.6. The three
commonest roles are Manager, Research Director and Sales Executive. Attrition is
**7.4%**, well below the 16.1% company average.

**Cluster 1 (1,144 employees)** is early-career staff: average age 34.7, average
monthly income 4,497 — under a third of Cluster 0's — and about 5 years at the
company at job level 1.6. The commonest roles are Research Scientist, Sales Executive
and Laboratory Technician. Attrition is **18.6%**, two and a half times Cluster 0's
rate.

The most striking result is what does *not* differ. Every satisfaction-related score i.e
job satisfaction, environment satisfaction, work-life balance, job involvement,
relationship satisfaction, sits at roughly 2.7 in both clusters. The two groups are
equally satisfied and have very different attrition rates.

The engineered features sharpen this. Income_Per_Job_Level is 3,700 in Cluster 0
against 2,767 in Cluster 1, so Cluster 1 is paid less even after accounting for their
lower seniority. Burnout_Risk and Company_Tenure_Ratio are near-identical across the
two.

The conclusion is that career stage and pay, not happiness, separate these groups.
An HR manager reading this should not be commissioning a satisfaction survey but should be looking at early-career pay
and progression.

### 3.7 Naming the clusters

A cluster number means nothing to the person who has to act on it, so each group gets
a plain-English name. The names describe **who the group is**, not what happens to
them.

In [ ]:
# A cluster number means nothing to an HR manager, so each group gets a
# plain-English name describing who they are - not what happens to them.
cluster_names = {
    0: 'Established Senior Staff',
    1: 'Early-Career Core Workforce',
}

summary = pd.DataFrame({
    'Name': pd.Series(cluster_names),
    'Employees': df_fe.groupby('Cluster').size(),
    'Avg age': df_fe.groupby('Cluster')['Age'].mean().round(1),
    'Avg monthly income': df_fe.groupby('Cluster')['MonthlyIncome'].mean().round(0),
    'Avg years at company': df_fe.groupby('Cluster')['YearsAtCompany'].mean().round(1),
    'Attrition rate (%)': (df_fe.groupby('Cluster')['Attrition'].mean() * 100).round(1),
})
summary

**Cluster 0 — Established Senior Staff.** Mid-forties, senior job levels, over a
decade at the company, the highest-paid group. Managers and Research Directors.

**Cluster 1 — Early-Career Core Workforce.** Mid-thirties, junior job levels, around
five years in, paid roughly a third of Cluster 0. Research Scientists, Sales
Executives and Laboratory Technicians — the bulk of the company.

## Part 4. Classification (20%)

Three classifiers plus a fourth for the Depth Track, all measured against a
deliberately unintelligent baseline.

Because the dataset has 1,470 rows, below the brief's 2,000 threshold, the main
evaluation is **stratified 5-fold cross-validation** rather than a single split. A
single 80/20 split would leave only about 47 leavers in the test set, few enough that
an unlucky split moves the score by several points. Cross-validation runs five
different splits and reports the mean and standard deviation, so the spread is visible
alongside the score.

### 4.1 Setup

`stratify=y` on the split, and `StratifiedKFold` for the folds, both keep the 16%
leaver rate intact in every portion of the data.

In [ ]:
# Part 4 imports
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import confusion_matrix, roc_curve, roc_auc_score

# A single split, used later for the confusion matrices.
# stratify=y keeps the 16% leaver rate the same in both halves; without it a
# random split could easily land far more leavers in one side than the other.
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled_df, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE)

# The main evaluation is cross-validation, because 1,470 rows is under the
# 2,000 threshold in the brief. Five folds, each preserving the 16% rate.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Train:', X_train.shape[0], 'employees,', int(y_train.sum()), 'leavers')
print('Test :', X_test.shape[0], 'employees,', int(y_test.sum()), 'leavers')
print('Leaver rate  train %.3f   test %.3f' % (y_train.mean(), y_test.mean()))

### 4.2 The models, and why each one

**Logistic Regression** — gives every column a weight and adds them up. Fully
readable: an HR manager can be told exactly which factors pushed a given employee's
risk up. Its limit is that it can only draw straight relationships.

**Decision Tree** — a sequence of yes/no questions, the only model here you could draw
on paper and hand to a manager. We expect it to memorise the training data rather than
generalise, and we include it partly to show what that looks like in numbers.

**Random Forest** — hundreds of trees built on different slices of the data, voting.
Individual mistakes tend to cancel out. Usually strong, but no longer readable.

**Gradient Boosting** (Depth Track fourth model) — trees built one after another, each
correcting the errors of the last.

**Stratified Dummy** — required by the brief. It ignores the data entirely and guesses
in proportion to the real 84/16 split. It is the floor every real model must clear.

In [ ]:
# Four classifiers plus the required baseline.
models = {
    'Stratified Dummy':    DummyClassifier(strategy='stratified', random_state=RANDOM_STATE),
    'Logistic Regression': LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    'Decision Tree':       DecisionTreeClassifier(random_state=RANDOM_STATE),
    'Random Forest':       RandomForestClassifier(n_estimators=200, random_state=RANDOM_STATE, n_jobs=-1),
    'Gradient Boosting':   GradientBoostingClassifier(random_state=RANDOM_STATE),
}
print('Models to compare:', len(models) - 1, 'classifiers plus the dummy baseline')

### 4.3 Cross-validated scoring

In [ ]:
# Score every model with 5-fold cross-validation.
# return_train_score lets us compare training performance against test
# performance, which is how overfitting becomes visible.
SCORING = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']

rows = []
for name, model in models.items():
    r = cross_validate(model, X_scaled_df, y, cv=cv, scoring=SCORING,
                       n_jobs=-1, return_train_score=True)
    row = {'Model': name}
    for s in SCORING:
        row[s + '_mean'] = r['test_' + s].mean()
        row[s + '_std'] = r['test_' + s].std()
    row['train_f1'] = r['train_f1'].mean()
    rows.append(row)

cv_results = pd.DataFrame(rows).set_index('Model')
print('Cross-validation complete.')

### 4.4 Comparison table

Models in rows, metrics in columns, each as mean ± standard deviation across the five
folds.

In [ ]:
# The comparison table the brief asks for: models in rows, metrics in columns,
# each shown as mean plus or minus standard deviation across the five folds.
comparison = pd.DataFrame(index=cv_results.index)
for s in SCORING:
    comparison[s.upper()] = (cv_results[s + '_mean'].map('{:.3f}'.format)
                             + ' ± ' + cv_results[s + '_std'].map('{:.3f}'.format))
comparison

**Reading the table**

Logistic Regression is the best model here, with F1 **0.532** and ROC AUC **0.833**.
This was not what we expected. The usual assumption is that Random Forest wins on
tabular data, and it did not.

Random Forest scored the **worst F1 of the three at 0.286**, despite a respectable ROC
AUC of 0.805. The reason is visible in the columns either side: precision 0.840,
recall 0.177. It is almost always right when it flags somebody, and it almost never
flags anybody. On an 84/16 split, predicting "stays" is the safe move, and the forest
takes it.

That gap between ROC AUC and F1 is worth understanding. ROC AUC asks whether the model
*ranks* leavers above stayers, and Random Forest does that well. F1 asks what happens
at the default 0.5 cut-off, and there it fails, because the threshold is wrong for
imbalanced data rather than the ranking being bad. Part 5 fixes exactly this.

Decision Tree scored F1 0.375 and ROC AUC 0.628. It is the only model that does not
reach the 0.75 ROC AUC floor.

Standard deviations are small for accuracy and ROC AUC (0.006–0.032), and larger for
precision, reaching ±0.142 for Random Forest. With roughly 47 leavers per fold, a
handful of predictions moves precision noticeably. This is precisely why the brief
requires cross-validation on a dataset this size.

### 4.5 How did our models do compared to the baseline model?

In [ ]:
# Checking explicitly
dummy = cv_results.loc['Stratified Dummy']
print('Stratified dummy baseline:  F1 %.3f   ROC AUC %.3f' % (dummy.f1_mean, dummy.roc_auc_mean))
print('-' * 72)
for name in cv_results.index:
    if name == 'Stratified Dummy':
        continue
    r = cv_results.loc[name]
    print('%-21s F1 %.3f %s | ROC AUC %.3f %s | AUC >= 0.75 %s' % (
        name, r.f1_mean, 'beats dummy' if r.f1_mean > dummy.f1_mean else 'FAILS     ',
        r.roc_auc_mean, 'beats dummy' if r.roc_auc_mean > dummy.roc_auc_mean else 'FAILS     ',
        'yes' if r.roc_auc_mean >= 0.75 else 'NO'))
print('-' * 72)
print('Highest ROC AUC seen: %.3f' % cv_results.roc_auc_mean.max())
print('Nothing is close to 1.0, so there is no hidden leaking column.')

All four classifiers beat the stratified dummy on both F1 and ROC AUC, so the
requirement is met. Three of the four also clear the 0.75 ROC AUC floor; the Decision
Tree does not, at 0.628. Either way, the model we carry forward is Logistic Regression at 0.833.

The highest score anywhere is 0.833, nowhere near 1.0, which confirms there is no
hidden leaking column feeding the answer to the models.

### 4.6 Which models memorised rather than learned

Comparing each model's score on data it trained on against data it has never seen.

In [ ]:
# Compare how well each model scores on data it trained on against data it
# has never seen. A large gap means the model memorised rather than learned.
overfit = pd.DataFrame({
    'F1 on training data': cv_results.train_f1.round(3),
    'F1 on unseen data': cv_results.f1_mean.round(3),
})
overfit['gap'] = (overfit['F1 on training data'] - overfit['F1 on unseen data']).round(3)
overfit.sort_values('gap', ascending=False)

**What this shows**

The Decision Tree scores a perfect **1.000 on training data and 0.375 on unseen data**
— a gap of 0.625. It has memorised all 1,470 employees rather than learning anything
general about them, which is the behaviour we included it to demonstrate. Random
Forest shows the same perfect training score with a 0.714 gap, though its averaging
keeps the ranking quality higher.

Logistic Regression has by far the smallest gap at 0.075. It is too simple to memorise,
and on a dataset this size we have an advantage rather than a limitation.

### 4.7 Confusion matrices

The cell below counts, for each model, how many of the 47 leavers in the test set it
actually caught.

In [ ]:
# Confusion matrices on the held-out test set.
# The number that matters here is the bottom-left cell: leavers the model
# said would stay.
fitted = {}
fig, axes = plt.subplots(1, 5, figsize=(22, 4))

for ax, (name, model) in zip(axes, models.items()):
    m = model.fit(X_train, y_train)
    fitted[name] = m
    cm = confusion_matrix(y_test, m.predict(X_test))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax,
                xticklabels=['stayed', 'left'], yticklabels=['stayed', 'left'])
    tn, fp, fn, tp = cm.ravel()
    ax.set_title('%s\ncaught %d of %d leavers' % (name, tp, tp + fn))
    ax.set_xlabel('predicted')
    ax.set_ylabel('actual')

plt.tight_layout()
plt.show()

print('Leavers in the test set:', int(y_test.sum()))
for name, m in fitted.items():
    tn, fp, fn, tp = confusion_matrix(y_test, m.predict(X_test)).ravel()
    print('  %-21s caught %2d  missed %2d  false alarms %2d' % (name, tp, fn, fp))

**What the confusion matrices show**

Out of 47 leavers in the test set:

- Logistic Regression caught **18**, missing 29, with 8 false alarms
- Decision Tree caught 17, but raised **36** false alarms to do it
- Gradient Boosting caught 16 with 8 false alarms
- Random Forest caught **5**, missing 42
- The dummy caught 6 with 43 false alarms

Random Forest catches 5 of 47 while scoring 86% accuracy. It is correct about almost
every employee and identifies almost none of the ones the company needs to know about,
which is why accuracy was set aside as a measure at the end of Part 1.

The best model still misses 29 of 47 at the default cut-off.

## Part 5. Imbalance Handling (10%)

16.12% of employees left, so this dataset is imbalanced and the techniques below
apply rather than the "roughly balanced" case in the brief. All three are used:
class weights, threshold tuning and SMOTE.

### 5.1 Why the best model still misses 29 of 47

Before changing anything, it is worth seeing where the model actually places people.
It does not output "leaves" or "stays" — it outputs a risk score between 0 and 1,
and a separate rule turns that score into a decision. By default the rule is: flag
anyone above 0.5.

In [ ]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import (precision_score, recall_score, f1_score,
                             precision_recall_curve)
from imblearn.over_sampling import SMOTE

# The model chosen in Part 4, refitted on the training split.
final_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
final_model.fit(X_train, y_train)
probs = final_model.predict_proba(X_test)[:, 1]

print('Where the model places people on its 0-to-1 risk scale:')
print('  Employees who actually left  - median score %.3f' % np.median(probs[y_test == 1]))
print('  Employees who actually stayed - median score %.3f' % np.median(probs[y_test == 0]))
print()
print('  Leavers scoring above the default 0.5 cut-off: %d of %d'
      % ((probs[y_test == 1] > 0.5).sum(), (y_test == 1).sum()))

**What this shows**

The employees who actually left have a median score of **0.337**. Those who stayed
have a median of 0.043, so the model is separating them clearly — leavers score
roughly eight times higher.

But 0.337 is below 0.5, so the default rule discards them. Only 18 of the 47 leavers
score above the cut-off.

The model is not failing to identify at-risk employees. The threshold applied to its
output is set for a balanced dataset, and ours is not one. A score of 0.337 is more
than double this company's 16% base rate, which in business terms is a clear warning,
and the default rule throws it away.

### 5.2 Technique 1: class weights

During training, each leaver is counted more heavily than each stayer, in proportion
to how rare they are. This removes the model's incentive to predict "stays" for
almost everybody.

In [ ]:
# Technique 1: class weights.
# Tells the model during training that each leaver counts for more than each
# stayer, in proportion to how rare they are, so it stops defaulting to "stays".
weighted = LogisticRegression(max_iter=2000, class_weight='balanced',
                              random_state=RANDOM_STATE)
weighted.fit(X_train, y_train)
probs_weighted = weighted.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, weighted.predict(X_test)).ravel()
print('Class weights: caught %d of %d leavers, with %d false alarms' % (tp, tp + fn, fp))
print('F1 %.3f   recall %.3f   precision %.3f'
      % (f1_score(y_test, weighted.predict(X_test)),
         recall_score(y_test, weighted.predict(X_test)),
         precision_score(y_test, weighted.predict(X_test))))

Class weights raise recall sharply, from 18 leavers caught to **32**. The cost
is precision, which falls from 0.692 to 0.368: the number of false alarms rises from
8 to 55. F1 drops slightly to 0.478 because the loss in precision outweighs the gain
in recall.

### 5.3 Technique 2: threshold tuning

The model is left untouched and only the cut-off moves. The threshold is selected
using the training data alone, through out-of-fold predictions, so the test set plays
no part in choosing it.

In [ ]:
# Technique 2: threshold tuning.
# The threshold is chosen using TRAINING data only, via out-of-fold predictions,
# so the test set stays untouched until the final evaluation.
oof_probs = cross_val_predict(
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    X_train, y_train, cv=cv, method='predict_proba')[:, 1]

prec, rec, thresh = precision_recall_curve(y_train, oof_probs)
f1_curve = 2 * prec[:-1] * rec[:-1] / (prec[:-1] + rec[:-1] + 1e-9)

fig, ax = plt.subplots(1, 2, figsize=(14, 4.5))
ax[0].plot(rec, prec, lw=2)
ax[0].set_xlabel('Recall - share of leavers caught')
ax[0].set_ylabel('Precision - share of flags that are correct')
ax[0].set_title('Precision-Recall curve (training data)')

ax[1].plot(thresh, f1_curve, lw=2, color='darkorange')
ax[1].axvline(0.30, ls='--', c='k', lw=1, label='chosen threshold 0.30')
ax[1].set_xlabel('Threshold')
ax[1].set_ylabel('F1')
ax[1].set_title('F1 at each threshold (training data)')
ax[1].legend()
plt.tight_layout()
plt.show()

print('Threshold with the highest F1 on training data: %.3f' % thresh[np.argmax(f1_curve)])
print('We select 0.30 instead, on the business grounds set out below.')

**Choosing the threshold on business cost**

The highest-F1 threshold on training data is 0.375. We did not take it, because F1
treats a missed leaver and a false alarm as equally bad and for this company they are
not remotely equal.

A **missed leaver** means somebody resigns: recruitment, notice period, onboarding,
and lost knowledge. A common rule of thumb is around six months of that person's
salary. The median leaver in this dataset earns 3,202 a month, so roughly 19,200 per
person.

A **false alarm** means a manager has a retention conversation with somebody who was
not going to leave. That costs an hour or so of management time, taken here as 500.

A miss therefore costs roughly forty times a false alarm. The table below prices each
threshold accordingly.

In [ ]:
# What each threshold would cost the company.
# Assumptions, stated so they can be challenged:
#   - Replacing somebody who leaves costs about 6 months of their salary
#     (recruitment, notice period, onboarding, lost knowledge).
#   - A retention conversation with somebody who was not going to leave costs
#     a manager's time, taken here as 500.
REPLACEMENT_MONTHS = 6
CONVERSATION_COST = 500

test_salaries = df_fe.loc[X_test.index, 'MonthlyIncome'].values

cost_rows = []
for t in [0.50, 0.40, 0.35, 0.30, 0.25, 0.20, 0.15, 0.10]:
    preds = (probs >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    missed_cost = REPLACEMENT_MONTHS * test_salaries[(y_test.values == 1) & (preds == 0)].sum()
    alarm_cost = CONVERSATION_COST * fp
    cost_rows.append({
        'Threshold': t, 'Caught': tp, 'Missed': fn, 'False alarms': fp,
        'Staff flagged': tp + fp,
        'Cost of misses': int(missed_cost), 'Cost of conversations': int(alarm_cost),
        'Total cost': int(missed_cost + alarm_cost),
        'F1': round(f1_score(y_test, preds), 3)})

cost_table = pd.DataFrame(cost_rows).set_index('Threshold')
print('Test set: %d employees, %d of them leavers' % (len(y_test), int(y_test.sum())))
cost_table

**Why 0.30 and not the cheapest option**

On cost alone, lower is better all the way down: 0.10 costs 452,392 against 891,574
at the default cut-off.

We chose **0.30**, which is not the cheapest row, for a reason the cost model does not
capture. At 0.10 the model flags 111 of 294 employees — 38% of the workforce — and the
calculation assumes every one of them receives a real retention conversation at 500. An
HR team cannot hold 111 meaningful conversations, and conversations held under that
kind of load stop being meaningful, so the modelled saving would not materialise.
There is also a risk the company would be creating the very problem it is trying to
solve: Part 1 found that overtime triples the quit rate, and overloading the HR team
is a way to lose HR staff.

At 0.30 the model flags 48 employees, catches 26 of the 47 leavers against 18 at the
default, and reduces modelled cost from 891,574 to 742,772. That is a workload a real
HR team can deliver on, which is what makes the saving credible.

### 5.4 Technique 3: SMOTE

SMOTE creates synthetic additional leavers until the training set is balanced. It is
applied to the training data only — generating synthetic neighbours before the split
would place near-copies of test employees into training and inflate every score that
followed.

In [ ]:
# Technique 3: SMOTE.
# Creates synthetic extra leavers until the training set is balanced.
# Applied to the TRAINING data only - generating synthetic neighbours before
# the split would leak information about test employees into training and
# inflate every score that follows.
smote = SMOTE(random_state=RANDOM_STATE)
X_resampled, y_resampled = smote.fit_resample(X_train, y_train)

print('Training set before SMOTE:', dict(y_train.value_counts()))
print('Training set after SMOTE :', dict(pd.Series(y_resampled).value_counts()))

smote_model = LogisticRegression(max_iter=2000, random_state=RANDOM_STATE)
smote_model.fit(X_resampled, y_resampled)
probs_smote = smote_model.predict_proba(X_test)[:, 1]

tn, fp, fn, tp = confusion_matrix(y_test, smote_model.predict(X_test)).ravel()
print()
print('SMOTE: caught %d of %d leavers, with %d false alarms' % (tp, tp + fn, fp))

SMOTE behaves much like class weights: 30 leavers caught, 51 false alarms.
Both techniques rebalance the training data, so it is unsurprising that they land in
a similar place.

### 5.5 Baseline versus each technique

In [ ]:
# Baseline against every technique, on the same held-out test set.
def score_row(name, preds, probabilities):
    tn, fp, fn, tp = confusion_matrix(y_test, preds).ravel()
    return {'Technique': name,
            'Precision': round(precision_score(y_test, preds, zero_division=0), 3),
            'Recall': round(recall_score(y_test, preds), 3),
            'F1': round(f1_score(y_test, preds), 3),
            'ROC AUC': round(roc_auc_score(y_test, probabilities), 3),
            'Caught': tp, 'Missed': fn, 'False alarms': fp}

dummy_final = DummyClassifier(strategy='stratified', random_state=RANDOM_STATE).fit(X_train, y_train)

imbalance_table = pd.DataFrame([
    score_row('Stratified dummy', dummy_final.predict(X_test), dummy_final.predict_proba(X_test)[:, 1]),
    score_row('Baseline, threshold 0.50', (probs >= 0.50).astype(int), probs),
    score_row('Class weights', weighted.predict(X_test), probs_weighted),
    score_row('SMOTE', smote_model.predict(X_test), probs_smote),
    score_row('Threshold 0.30 (chosen)', (probs >= 0.30).astype(int), probs),
]).set_index('Technique')
imbalance_table

**What the table shows**

Every technique raises recall above the 0.383 baseline. They differ in what they
charge for it.

Class weights and SMOTE buy the most recall (0.681 and 0.638) but flag 55 and 51
people wrongly, dropping precision below 0.37 and F1 below the untouched baseline.

Threshold tuning at 0.30 gives the highest F1 at **0.547**, catching 26 leavers with
22 false alarms. It is the only technique that improves F1 over the baseline rather
than trading it away.

ROC AUC is 0.813 for the baseline, the threshold version and class weights, and
0.802 for SMOTE. Threshold tuning does not change ROC AUC at all, because moving the
cut-off does not change how the model ranks people — it only changes where the line
is drawn through that ranking.

### 5.6 Final model

The numbers below are cross-validated across all 1,470 rows, matching how our Part 4 results, and they are the figures given in the Submission Summary.

In [ ]:
# Final check on all 1,470 rows using cross-validation, since the dataset is
# under 2,000 rows. These are the numbers reported in the Submission Summary.
oof_all = cross_val_predict(
    LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    X_scaled_df, y, cv=cv, method='predict_proba')[:, 1]

oof_dummy = cross_val_predict(
    DummyClassifier(strategy='stratified', random_state=RANDOM_STATE),
    X_scaled_df, y, cv=cv, method='predict_proba')[:, 1]

final_preds = (oof_all >= 0.30).astype(int)
dummy_preds = (oof_dummy >= 0.50).astype(int)

print('FINAL MODEL: Logistic Regression, decision threshold 0.30')
print('-' * 58)
print('                       F1       ROC AUC')
print('Stratified dummy     %.3f       %.3f' % (f1_score(y, dummy_preds), roc_auc_score(y, oof_dummy)))
print('Final model          %.3f       %.3f' % (f1_score(y, final_preds), roc_auc_score(y, oof_all)))
print('-' * 58)
print('Beats dummy on F1      :', f1_score(y, final_preds) > f1_score(y, dummy_preds))
print('Beats dummy on ROC AUC :', roc_auc_score(y, oof_all) > roc_auc_score(y, oof_dummy))
print('ROC AUC at least 0.75  :', roc_auc_score(y, oof_all) >= 0.75)
print()
print('Recall %.3f   precision %.3f' % (recall_score(y, final_preds), precision_score(y, final_preds)))

**Final model: Logistic Regression with a decision threshold of 0.30.**

Cross-validated F1 **0.567** and ROC AUC **0.832**, against the stratified dummy's
0.141 and 0.486. Both hard requirements are met, and ROC AUC is well above the 0.75
floor. Recall is 0.608, so the model identifies about three in five of the employees
who go on to resign, compared with two in five at the default cut-off.

### Run time


In [ ]:
elapsed = time.time() - RUN_START
print('Full notebook run time: %.1f minutes' % (elapsed / 60))
conn.close()

## Part 6. The Tie-In (5%)


It computes the attrition rate inside each K-Means cluster from Part 3, and answers
whether any cluster turned out to be a strong predictor of who resigns, and what that
says about the data.

## Depth Track (10 bonus marks)

**Classification Depth:** four classifiers with cross-validation, all ROC curves on one
chart, and a comparison of which columns each model relied on.

### D.1 All four ROC curves on one chart

A curve nearer the top-left corner catches more leavers for fewer false alarms.

In [ ]:
# All four ROC curves on one chart. A curve closer to the top-left corner
# is a model that catches more leavers for fewer false alarms.
plt.figure(figsize=(7.5, 6.5))

for name, m in fitted.items():
    if name == 'Stratified Dummy':
        continue
    probs = m.predict_proba(X_test)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, probs)
    plt.plot(fpr, tpr, lw=2, label='%s (AUC %.3f)' % (name, roc_auc_score(y_test, probs)))

plt.plot([0, 1], [0, 1], 'k--', lw=1, label='Random guessing (AUC 0.500)')
plt.xlabel('False alarm rate')
plt.ylabel('Share of leavers caught')
plt.title('ROC curves, held-out test set')
plt.legend(loc='lower right')
plt.show()

Logistic Regression and Gradient Boosting track each other closely across the whole
range. Random Forest sits slightly below despite its poor F1, which again shows that its
problem is the cut-off rather than the ranking. The Decision Tree's curve is visibly
angular and closest to the diagonal, because a single tree produces only a handful of
distinct probability values rather than a smooth range.

### D.2 Which columns did each model rely on?

The four models express importance in different units, so comparing raw values would be
meaningless. We compare **rankings** instead: rank 1 is the column that model leaned on
hardest, out of 46.

In [ ]:
# Which columns did each model rely on? The four models measure this in
# different units, so we compare their RANKINGS rather than their raw values.
# Rank 1 = the column that model leaned on most, out of 46.

importance = pd.DataFrame({
    'Logistic Regression': pd.Series(np.abs(fitted['Logistic Regression'].coef_[0]), index=X_scaled_df.columns),
    'Decision Tree':       pd.Series(fitted['Decision Tree'].feature_importances_, index=X_scaled_df.columns),
    'Random Forest':       pd.Series(fitted['Random Forest'].feature_importances_, index=X_scaled_df.columns),
    'Gradient Boosting':   pd.Series(fitted['Gradient Boosting'].feature_importances_, index=X_scaled_df.columns),
})
ranks = importance.rank(ascending=False).astype(int)

top15 = importance['Random Forest'].sort_values(ascending=False).head(15).index
print('IMPORTANCE RANK ACROSS MODELS (1 = most important of 46 columns)')
print(ranks.loc[top15].to_string())

### D.3 Where the models disagree, and what that reveals

In [ ]:
# Where the four models disagree most about what matters.
spread = (ranks.max(axis=1) - ranks.min(axis=1)).sort_values(ascending=False)
print('BIGGEST DISAGREEMENTS')
print(pd.concat([spread.head(6).rename('rank spread'), ranks.loc[spread.head(6).index]], axis=1).to_string())
print()

# The three Rate columns are the heart of the disagreement, so test them directly.
print('DO THE RATE COLUMNS ACTUALLY RELATE TO LEAVING?')
for c in ['DailyRate', 'HourlyRate', 'MonthlyRate', 'MonthlyIncome']:
    bands = pd.qcut(df_fe[c], 4)
    quit_by_band = df_fe.groupby(bands, observed=True)['Attrition'].mean().mul(100).round(1).tolist()
    print('  %-14s quit%% by quartile: %-28s correlation with leaving: %+.3f'
          % (c, str(quit_by_band), df_fe[c].corr(df_fe['Attrition'])))
print()
print('And how the three engineered features from Part 2 ranked:')
print(ranks.loc[['Income_Per_Job_Level', 'Burnout_Risk', 'Company_Tenure_Ratio']].to_string())

**The main finding**

All four models rank MonthlyIncome, TotalWorkingYears and Age near the top, consistent
with the Part 1 findings.

They disagree sharply elsewhere. `HourlyRate` ranks **46th out of 46 for Logistic
Regression and 6th for the Decision Tree**. `MonthlyRate` ranks 44th and
8th. These are the two largest disagreements in the whole comparison, so we tested the
columns directly.

The test settles it. `HourlyRate` has a correlation with leaving of **−0.007** and quit
rates of 15.2%, 18.0%, 14.6% and 16.6% across its four quartiles — flat. `MonthlyRate`
is the same story at +0.015. For comparison, `MonthlyIncome` runs 29.3% down to 10.3%
with a correlation of −0.160. The Rate columns carry no signal about attrition at all.

So the tree-based models are ranking two noise columns among their top eight. The reason
is a known weakness of how tree importance is calculated: a column with many distinct
values offers a tree many possible places to split, and some of those splits will
separate the training data well by chance. `MonthlyRate` has 1,427 distinct values across
1,470 employees, so it offers enormous opportunity to fit noise. Logistic Regression
cannot do this — it fits one weight per column and correctly assigns these near-zero
weights.

Reported on its own, the Random Forest importance ranking would have told the HR manager
that hourly rate is among the top drivers of resignation. The quit rates above show it is
not. Only the comparison across four models exposed this; any single model would have
concealed it.

**Our engineered features.** `Burnout_Risk` ranks **1st for Gradient Boosting and 4th for
Random Forest**, so the interaction term built in Part 2 is genuinely doing work.
`Income_Per_Job_Level` ranks consistently in the 6th-to-13th band across all four models,
the steadiest showing of the three. `Company_Tenure_Ratio` ranks 5th for the Decision Tree
but 30th for Logistic Regression, consistent with the uneven pattern noted in Part 2.

The best-scoring model is also the simplest and the only one whose reasoning can be shown
to a manager: Logistic Regression assigns one readable weight per column, so any individual
prediction can be traced. The two ensemble models score lower here and cannot be traced.